# Model Training, Tuning, and Experiment Tracking
This notebook demonstrates how the end-to-end model training, hyperparameter tuning, evaluation, and experiment tracking pipeline runs.

We use **MLflow** to log parameters, metrics, confusion matrix, ROC curve, and the serialized scikit-learn models.

In [ ]:
import os
import sys

# Add the root directory to path to enable src imports
root_dir = os.path.dirname(os.getcwd())
if root_dir not in sys.path:
    sys.path.append(root_dir)

import mlflow
import pandas as pd
import joblib

from src.preprocess import load_and_clean_data
from src.train import train_and_evaluate

## 1. Set up data paths

In [ ]:
data_path = os.path.join(root_dir, "data", "raw", "processed.cleveland.data")
print(f"Dataset exists: {os.path.exists(data_path)}")

## 2. Train Logistic Regression Model
We run the training module for the **Logistic Regression** baseline. Hyperparameters are tuned via 5-fold cross-validation on the grid.

In [ ]:
lr_pipeline, lr_metrics = train_and_evaluate(data_path, model_type="lr")
print("Logistic Regression Test Metrics:")
for k, v in lr_metrics.items():
    print(f"  {k}: {v:.4f}")

## 3. Train Random Forest Model
We train the **Random Forest** complex classifier on the same features, logging the parameters, curves, and artifacts to MLflow.

In [ ]:
rf_pipeline, rf_metrics = train_and_evaluate(data_path, model_type="rf")
print("Random Forest Test Metrics:")
for k, v in rf_metrics.items():
    print(f"  {k}: {v:.4f}")

## 4. Run Model Selection & Save Best Model
We select the model that maximizes **ROC-AUC** on the test set, and serialize it for FastAPI production serving.

In [ ]:
best_model_path = os.path.join(root_dir, "models", "best_model.joblib")

if rf_metrics["roc_auc"] > lr_metrics["roc_auc"]:
    print("Random Forest is the best performing model. Saving to production path...")
    best_pipeline = rf_pipeline
else:
    print("Logistic Regression is the best performing model. Saving to production path...")
    best_pipeline = lr_pipeline
    
os.makedirs(os.path.dirname(best_model_path), exist_ok=True)
joblib.dump(best_pipeline, best_model_path)
print(f"Best model pipeline successfully saved to: {best_model_path}")

## 5. Test Inference Pipeline on Sample Input

In [ ]:
# Create a single mock patient input matching the schema
sample_data = pd.DataFrame([{
    "age": 63.0,
    "sex": 1.0,
    "cp": 3.0,
    "trestbps": 145.0,
    "chol": 233.0,
    "fbs": 1.0,
    "restecg": 0.0,
    "thalach": 150.0,
    "exang": 0.0,
    "oldpeak": 2.3,
    "slope": 2.0,
    "ca": 0.0,
    "thal": 3.0
}])

# Load serialized model
production_model = joblib.load(best_model_path)

# Run preprocessing and inference end-to-end
pred = production_model.predict(sample_data)[0]
prob = production_model.predict_proba(sample_data)[0][1]

print(f"Prediction (1=Disease, 0=No Disease): {pred}")
print(f"Confidence score: {prob:.4f}")

## 6. How to View Tracked Experiments (MLflow UI)

To launch the MLflow UI local web server and review runs, parameter parameters, comparative metric tables, and logged ROC curves, open your terminal in the project directory and run:
```bash
mlflow ui
```
Then open **http://127.0.0.1:5000** in your browser.